In [ ]:
import duckdb
import os

db_path = '../../data/duckdb/mimic.db'
base_csv_path = "../../data/mimic-iv-3.1/icu/"

con = duckdb.connect(db_path)

icu_tables = {
    'v_caregiver': 'caregiver.csv.gz',
    'v_chartevents': 'chartevents.csv.gz',
    'v_d_items': 'd_items.csv.gz',
    'v_datetimeevents': 'datetimeevents.csv.gz',
    'v_icustays': 'icustays.csv.gz',
    'v_ingredientevents': 'ingredientevents.csv.gz',
    'v_inputevents': 'inputevents.csv.gz',
    'v_outputevents': 'outputevents.csv.gz',
    'v_procedureevents': 'procedureevents.csv.gz'
}

print("Début de l'importation des données")

for table_name, file_name in icu_tables.items():
    full_path = os.path.join(base_csv_path, file_name)
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM '{full_path}'")


print("\nData sucessfully imported. Current tables in the database:")
print(con.execute("SHOW TABLES").df())
con.close()

Looking for % of NA and extracting a csv

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('../../data/duckdb/mimic.db')

def get_full_na_report(tables):
    all_reports = []
    
    for table in tables:
        print(f"Analysis of the table : {table}...")
        try:
            cols_info = con.execute(f"DESCRIBE {table}").df()
            columns = cols_info['column_name'].tolist()
            select_parts = [f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in columns]
            sql = f'SELECT COUNT(*) as total_rows, {", ".join(select_parts)} FROM {table}'
            
            res = con.execute(sql).df()
            total_rows = res['total_rows'][0]
            
            df_na = res.drop(columns=['total_rows']).T
            df_na.columns = ['na_count']
            df_na['table'] = table
            df_na['variable'] = df_na.index
            df_na['pct_na'] = (df_na['na_count'] / total_rows * 100).round(2)
            
            all_reports.append(df_na[['table', 'variable', 'na_count', 'pct_na']])
        except Exception as e:
            print(f"Erreur sur {table}: {e}")

    final_report = pd.concat(all_reports).reset_index(drop=True)
    return final_report

icu_views = [
    'v_caregiver', 'v_chartevents', 'v_d_items', 
    'v_datetimeevents', 'v_icustays', 'v_ingredientevents', 
    'v_inputevents', 'v_outputevents', 'v_procedureevents'
]

full_report = get_full_na_report(icu_views)

full_report.to_csv('mimic_na_report.csv', index=False)
print("\n The complete report is exported in 'mimic_na_report.csv'")

# Generating the Cleaned Dataset

Following the statistical analysis of missing values, we have implemented a **Feature Selection** and **Data Cleaning** pipeline. The objective is to transition from raw clinical data to an optimized analytical dataset by removing noise and technical artifacts while preserving essential clinical signals.

### Cleaning Strategy & Justification

We applied the following rules to the initial 113+ variables across the ICU tables:

1. **Missingness Threshold (>70%)**: 
   Variables with a high percentage of missing data, such as `location`, `locationcategory` (Procedures), and `lownormalvalue`/`highnormalvalue` (Items), were excluded. These columns lack sufficient information density for robust statistical modeling.
   
2. **Technical Metadata Removal**: 
   Columns like `storetime` and `row_id` were dropped. These are administrative logs indicating when data was entered into the system rather than when the clinical event occurred (`charttime`). Removing them prevents **data leakage** and reduces dimensionality.

3. **Redundancy Filtering**: 
   Units of measurement (`valueuom`) were removed in cases where the measurement type is already uniquely identified by the `itemid`, simplifying the categorical feature space.

4. **Clinical Preservation (Manual Overrides)**:
   * **`valuenum`**: Despite a ~61% missing rate in `chartevents`, this column is retained as it contains the primary numerical values for vital signs and lab results.
   * **`rate`**: Retained in infusion tables (`inputevents`/`ingredientevents`). In clinical practice, a `NULL` rate often signifies a **bolus** (one-time dose) rather than a missing data point.

In [ ]:
raw_db_path = '../../data/duckdb/mimic.db'
con = duckdb.connect(raw_db_path)

print("Step 1: Filtering columns based on NA threshold...")

con.execute("CREATE OR REPLACE TABLE temp_chartevents AS SELECT subject_id, hadm_id, stay_id, caregiver_id, charttime, itemid, value, valuenum, warning FROM v_chartevents")
con.execute("CREATE OR REPLACE TABLE temp_d_items AS SELECT itemid, label, abbreviation, linksto, category, param_type FROM v_d_items")
con.execute("CREATE OR REPLACE TABLE temp_procedureevents AS SELECT * EXCLUDE (storetime, location, locationcategory) FROM v_procedureevents")
con.execute("CREATE OR REPLACE TABLE temp_inputevents AS SELECT * EXCLUDE (storetime) FROM v_inputevents")
con.execute("CREATE OR REPLACE TABLE temp_ingredientevents AS SELECT * EXCLUDE (storetime) FROM v_ingredientevents")
con.execute("CREATE OR REPLACE TABLE temp_icustays AS SELECT * FROM v_icustays")
con.execute("CREATE OR REPLACE TABLE temp_outputevents AS SELECT * EXCLUDE (storetime) FROM v_outputevents")
con.execute("CREATE OR REPLACE TABLE temp_datetimeevents AS SELECT * EXCLUDE (storetime) FROM v_datetimeevents")

In [ ]:
print("\nStep 2: Analyzing Variance and generating CSV...")

variance_results = []
tables_to_check = [
    'temp_chartevents', 
    'temp_inputevents', 
    'temp_outputevents', 
    'temp_icustays',
    'temp_procedureevents', 
    'temp_datetimeevents', 
    'temp_ingredientevents'
]
variance_threshold = 0.0001 

for table in tables_to_check:
    cols_info = con.execute(f"PRAGMA table_info('{table}')").df()
    num_cols = cols_info[cols_info['type'].str.contains('INT|DOUBLE|FLOAT|DECIMAL', case=False)]['name'].tolist()
    
    features = [c for c in num_cols if not c.endswith('_id') and c != 'itemid']
    
    if features:
        var_query = ", ".join([f"VARIANCE({col}) AS {col}" for col in features])
        variances = con.execute(f"SELECT {var_query} FROM {table}").df().iloc[0]
        
        for col, val in variances.items():
            status = "Keep" if val >= variance_threshold else "Drop (Low Variance)"
            variance_results.append({'table': table, 'column': col, 'variance': val, 'status': status})
            
            if val < variance_threshold:
                print(f"  > Dropping {col} from {table} (Variance: {val:.6f})")
                con.execute(f"ALTER TABLE {table} DROP COLUMN {col}")

df_var_report = pd.DataFrame(variance_results)
df_var_report.to_csv('./variance_report.csv', index=False)
print(f" Variance report saved to: ./variance_report.csv")

In [ ]:
import duckdb
import os

managed_db_path = '../../data/duckdb/mimic_managed.db'

print(f"\nStep 3: Exporting final dataset to {managed_db_path}...")

if os.path.exists(managed_db_path): 
    os.remove(managed_db_path)

con.execute(f"ATTACH '{managed_db_path}' AS mng_db")

temp_tables = con.execute("SHOW TABLES").df()['name'].tolist()

for table in temp_tables:
    if table.startswith('temp_'):
        clean_name = table.replace('temp_', 'clean_')
        print(f"  > Saving {clean_name} to permanent storage...")
        
        con.execute(f"CREATE TABLE mng_db.{clean_name} AS SELECT * FROM {table}")

con.execute("DETACH mng_db")
con.close()

print("\n'mimic_managed.db' is now saved ")

# extract the data base

In [ ]:
import duckdb
import os

managed_db_path = '../../data/duckdb/mimic_managed.db'
con = duckdb.connect(managed_db_path)

export_path = '../../data/duckdb/csv_exports'
os.makedirs(export_path, exist_ok=True)

tables = con.execute("SHOW TABLES").df()['name'].tolist()

for table in tables:
    if table.startswith('clean_'):
        output_file = os.path.join(export_path, f"{table}.csv")
        print(f"   > Exporting {table}...")
        
        con.execute(f"""
            COPY {table} TO '{output_file}' 
            (HEADER, DELIMITER ',')
        """)

con.close()
print("\nFinished")